# Lesson 10: Combine causal heads with multi-head attention

A single head learns one set of query, key, and value projections. Multi-head attention runs several independent heads on the same input, concatenates their outputs, then uses a learned projection to mix the channels back to the model width.

Here we use two heads of size 2, so concatenation changes the last dimension from 4 to 4. The sizes are intentionally small; the important idea is that each head has its own parameters and can learn a different attention pattern.

The matching [C implementation](../c/src/multi_head_attention_main.c), [Java implementation](../java/src/main/java/learning/karpathy/gpt1/MultiHeadAttentionLesson.java), and [lesson map](../README.md#run-lessons-by-step) use the same input and fixed weights.

## Set up two independent heads

Each head repeats the causal attention calculation from Lesson 9 with its own Q/K/V projections. Karpathy's implementation stores the heads in a ModuleList, which registers each head's parameters with the larger module.

The source also applies dropout to attention weights and to the projected output. We include both with p=0.2. The main comparisons use evaluation mode, where dropout is inactive, so Python, C, and Java can be compared deterministically.

In [ ]:
import math

import torch
from torch import nn
from torch.nn import functional as F

torch.set_printoptions(precision=4, sci_mode=False)
torch.manual_seed(1337)

BATCH_SIZE = 2
TIME_STEPS = 4
CHANNELS = 4
NUM_HEADS = 2
HEAD_SIZE = 2
DROPOUT = 0.2

x = torch.tensor(
    [
        [
            [1.0, 0.0, 2.0, -1.0],
            [0.0, 1.0, 1.0, 2.0],
            [2.0, -1.0, 0.0, 1.0],
            [1.0, 2.0, -1.0, 0.0],
        ],
        [
            [0.0, 2.0, 1.0, -1.0],
            [1.0, -1.0, 2.0, 0.0],
            [-1.0, 0.0, 1.0, 2.0],
            [2.0, 1.0, 0.0, -1.0],
        ],
    ],
    dtype=torch.float64,
)

class CausalAttentionHead(nn.Module):
    def __init__(self, channels: int, head_size: int, max_time: int, dropout: float) -> None:
        super().__init__()
        self.query = nn.Linear(channels, head_size, bias=False, dtype=torch.float64)
        self.key = nn.Linear(channels, head_size, bias=False, dtype=torch.float64)
        self.value = nn.Linear(channels, head_size, bias=False, dtype=torch.float64)
        self.attention_dropout = nn.Dropout(dropout)
        mask = torch.tril(torch.ones(max_time, max_time, dtype=torch.bool))
        self.register_buffer("causal_mask", mask)

    def forward(self, inputs: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        time = inputs.shape[1]
        queries = self.query(inputs)
        keys = self.key(inputs)
        values = self.value(inputs)
        scores = queries @ keys.transpose(-2, -1) / math.sqrt(queries.shape[-1])
        scores = scores.masked_fill(~self.causal_mask[:time, :time], float("-inf"))
        weights = self.attention_dropout(F.softmax(scores, dim=-1))
        return weights @ values, weights

class MultiHeadSelfAttention(nn.Module):
    def __init__(
        self, channels: int, num_heads: int, head_size: int, max_time: int, dropout: float
    ) -> None:
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttentionHead(channels, head_size, max_time, dropout) for _ in range(num_heads)]
        )
        self.projection = nn.Linear(num_heads * head_size, channels, dtype=torch.float64)
        self.projection_dropout = nn.Dropout(dropout)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        head_outputs = [head(inputs)[0] for head in self.heads]
        concatenated = torch.cat(head_outputs, dim=-1)
        return self.projection_dropout(self.projection(concatenated))

multi_head = MultiHeadSelfAttention(
    CHANNELS, NUM_HEADS, HEAD_SIZE, TIME_STEPS, DROPOUT
)

query_weights = [
    [[0.5, -0.5, 1.0, 0.0], [0.0, 1.0, 0.5, -0.5]],
    [[0.0, 0.5, -0.5, 1.0], [1.0, -0.5, 0.0, 0.5]],
]
key_weights = [
    [[1.0, 0.0, 0.5, -0.5], [-0.5, 0.5, 0.0, 1.0]],
    [[0.5, 1.0, -0.5, 0.0], [-1.0, 0.0, 0.5, 0.5]],
]
value_weights = [
    [[0.5, 0.0, 1.0, 0.5], [0.0, 1.0, -0.5, 0.5]],
    [[1.0, 0.5, 0.0, -0.5], [0.5, -1.0, 1.0, 0.0]],
]
projection_weight = [
    [0.5, 0.0, 0.5, 0.0],
    [0.0, 0.5, 0.0, 0.5],
    [0.5, 0.0, -0.5, 0.0],
    [0.0, 0.5, 0.0, -0.5],
]
projection_bias = [0.1, -0.2, 0.05, 0.3]

with torch.no_grad():
    for index, head in enumerate(multi_head.heads):
        head.query.weight.copy_(torch.tensor(query_weights[index], dtype=torch.float64))
        head.key.weight.copy_(torch.tensor(key_weights[index], dtype=torch.float64))
        head.value.weight.copy_(torch.tensor(value_weights[index], dtype=torch.float64))
    multi_head.projection.weight.copy_(torch.tensor(projection_weight, dtype=torch.float64))
    multi_head.projection.bias.copy_(torch.tensor(projection_bias, dtype=torch.float64))

multi_head.eval()
print("Input shape:", tuple(x.shape))
print("Head count:", len(multi_head.heads))
print("Dropout probability during training:", DROPOUT)

## Step 1: Run each head independently

Both heads see the same input, but each owns separate projection weights. Their causal attention patterns and outputs can therefore differ. Each head returns shape (batch, time, head_size) = (2, 4, 2).

In [ ]:
head_results = [head(x) for head in multi_head.heads]

for index, (head_output, attention_weights) in enumerate(head_results):
    print(f"Head {index} output shape:", tuple(head_output.shape))
    print(f"Head {index} attention weights for batch 0:\n", attention_weights[0])
    print(f"Head {index} output for batch 0:\n", head_output[0])

## Step 2: Concatenate head outputs

Concatenation happens along the last, channel dimension. For each token, head 0 contributes its two channels, followed by head 1's two channels. With two heads of size 2, this gives (batch, time, 4).

In [ ]:
head_outputs = [output for output, _weights in head_results]
concatenated = torch.cat(head_outputs, dim=-1)

print("Concatenated shape:", tuple(concatenated.shape))
print("Head 0 then head 1, batch 0:\n", concatenated[0])
assert concatenated.shape == (BATCH_SIZE, TIME_STEPS, NUM_HEADS * HEAD_SIZE)

## Step 3: Project back to the model width

The concatenated head channels are passed through a learned linear projection. It lets each output channel combine information from all heads. In Karpathy's code this projection is followed by dropout; here it is inactive because the module is in evaluation mode.

In [ ]:
projected = multi_head.projection(concatenated)
output = multi_head.projection_dropout(projected)
module_output = multi_head(x)

print("Projection weight shape:", tuple(multi_head.projection.weight.shape))
print("Projected output shape:", tuple(output.shape))
print("Multi-head output for batch 0:\n", output[0])
torch.testing.assert_close(module_output, output)
assert output.shape == x.shape

## Step 4: Check causality and see dropout's mode change

Each head has a causal mask, so concatenating and projecting their outputs cannot introduce future information. Then we switch to training mode and run twice with different random seeds. Dropout randomly removes some attention weights and projected values during training; in evaluation mode it is an identity operation.

In [ ]:
changed_x = x.clone()
changed_x[0, 3, 0] += 100.0
changed_output = multi_head(changed_x)
earlier_difference = (output[0, :3] - changed_output[0, :3]).abs().max()
print(f"Change at earlier positions: {earlier_difference.item():.12f}")
torch.testing.assert_close(output[0, :3], changed_output[0, :3], rtol=0, atol=1e-12)

multi_head.train()
torch.manual_seed(123)
training_output_a = multi_head(x)
torch.manual_seed(456)
training_output_b = multi_head(x)
dropout_difference = (training_output_a - training_output_b).abs().max()
print(f"Difference between training-mode runs: {dropout_difference.item():.6f}")
assert dropout_difference > 1e-8
multi_head.eval()
print("Causality check passed; dropout changes training-mode outputs and is inactive in evaluation mode.")

## What to notice

More heads do not mean that every head looks at a different fixed part of the sequence. Each head learns its own query, key, and value projections, so it can form a different data-dependent view of the same token sequence. Concatenation preserves each head's output channels; the final projection mixes them and restores the model width.

The C and Java examples show the deterministic evaluation-mode forward pass with fixed weights. They keep the attention math explicit and leave stochastic dropout to the Python library abstraction. The next lesson will combine this communication layer with a feedforward network, residual connections, and LayerNorm.

References: [PyTorch ModuleList](https://docs.pytorch.org/docs/stable/generated/torch.nn.ModuleList.html), [PyTorch Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html), [PyTorch Dropout](https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html), and Karpathy's [original notebook](../gpt_dev.ipynb).